# 從 GSM8K 題目，到一次 GeoRA 的 GRPO 更新

這份 notebook 跟著程式的資料流學習：

**題目 → 模型回答 → 答案檢查 → reward → 組內 advantage → token 概率比 → loss → A/B 更新。**

我們沿用已驗證的 difference 計算：

$$
W_{\mathrm{eff}}=W_{\mathrm{pre}}+c(BA-B_0A_0),\qquad c=\alpha/r.
$$

`W_pre` 是冻结的原模型權重；`A,B` 是準備更新的 FP32 參數；`A0,B0` 是冻结的 FP32 初值。
最開始 `A=A0, B=B0`，修正量為零。**本份教材不重新推導 mask/SVD 初始化。**

本機只載入小資料和隨機初始化的 tiny Qwen。人工 reward 的機械檢查與 Setonix 的真實 GSM8K 報告分開呈現；沒有載入 1.5B 模型，也沒有在本機訓練它。

GRPO 公式參考 [DeepSeekMath §4.1，Eq.3–4](https://arxiv.org/html/2402.03300v3#S4.SS1)。我們的 smoke 超參數及抽樣／打分路徑是明確記錄的復現選擇，不宣稱作者公開了這些全部細節。


## 1. 先確認我們使用哪個環境

使用此倉庫的 `.venv` 作為 notebook kernel。下個 cell 只找到倉庫、導入工具、限制 CPU 為 2 threads。
`PROJECT_DIRECTORY` 是 source 所在位置；`DATA_DIRECTORY` 是本機小資料的位置；這些與 Setonix 的 scratch 是不同路徑。


In [1]:
import copy
import json
import math
import os
from pathlib import Path
import sys

import torch
from transformers import Qwen2Config, Qwen2ForCausalLM

candidate_directories = [Path.cwd(), *Path.cwd().parents]
PROJECT_DIRECTORY = next(
    directory for directory in candidate_directories
    if (directory / "geora_layers.py").exists()
)
sys.path.insert(0, str(PROJECT_DIRECTORY))
sys.path.insert(0, str(PROJECT_DIRECTORY / "scripts"))

from gsm8k_data import build_prompt, load_prepared_rows, prepare_dataset, score_answer
from grpo_math import completion_token_mask, group_advantages, grpo_loss
from geora_layers import export_adapter_state, install_geora, load_geora_state

torch.set_num_threads(2)
DATA_DIRECTORY = Path(os.environ.get("GEORA_DATA_DIR", PROJECT_DIRECTORY / "datasets" / "gsm8k"))
print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Project:", PROJECT_DIRECTORY)
print("Dataset:", DATA_DIRECTORY)


Python: /Users/tianbaochen/geora-repro/.venv/bin/python3
PyTorch: 2.14.1
Project: /Users/tianbaochen/geora-repro
Dataset: /Users/tianbaochen/geora-repro/datasets/gsm8k


## 2. 模型拿到什麼？檢查器拿到什麼？

GSM8K 的一條資料包含問題和參考答案。生成模型只讀**問題＋固定作答指示**；答案檢查器另外保留參考的最後數值。

`question`：文字。`prompt`：真正發給模型的文字。`row['target']`：檢查器稍後使用的標準數值，沒有加入 `prompt`。

資料未準備時，下面只下載固定版本的原始 GSM8K 小檔案（數 MB），驗證 hash，並產生固定 split。
第一次執行需網路；資料已存在時驗證後重用。這裡不下載模型。


In [2]:
dataset_manifest = prepare_dataset(DATA_DIRECTORY)
smoke_rows = load_prepared_rows(DATA_DIRECTORY, "smoke")
row = smoke_rows[0]
question = row["question"]
prompt = build_prompt(question)

print("Dataset commit:", dataset_manifest["source_commit"])
print("Question ID:", row["id"])
print("Prompt sent to model:\n" + prompt)
# The prompt builder receives only the question, never the target or solution.
assert prompt == build_prompt(row["question"])


Dataset commit: 3101c7d5072418e28b9008a6636bde82a006892c
Question ID: train:01480
Prompt sent to model:
Solve the following math problem. Explain your reasoning, then end your answer with a final line in the form #### <number>. Write only the numeric answer after ####, without units.

Question:
Oli's banana split has 4 scoops of ice cream while Victoria has twice more scoops of ice cream than Oli's. How many more scoops of ice cream does Victoria have than Oli?


## 3. 一個回答的概率，究竟記的是什麼？

固定題目 $x$，某次回答的 token 依次是 $y_1,\ldots,y_T$。
第 $t$ 步模型讀 `x` 和已生成的前文，對**整個 vocab**輸出 logits；softmax 後取得下一 token 的分布。

$$
\log p(y\mid x)=\sum_{t=1}^{T}\log p(y_t\mid x,y_{<t}).
$$

**抽樣時的 logp** 是實際用來抽這個 token 的分布中，其概率的 log。
**重新打分的 logp** 是把已抽出的固定 token 放回模型，重新算其 log probability。它不要求重新生成回答。

下面只是把兩個步驟的「整個 vocab 分布 → 所抽 token 的 logp」攤開；這不是 1.5B 模型的真實回答。


In [3]:
# Two positions, each with logits over a three-token vocabulary.
logits_at_positions = torch.tensor([[1.0, 2.0, 3.0], [2.0, 0.0, 1.0]])
vocab_probabilities = torch.softmax(logits_at_positions.float(), dim=-1)
sampled_token_ids = torch.tensor([2, 0])
selected_log_probs = torch.log_softmax(logits_at_positions.float(), dim=-1).gather(
    dim=1,
    index=sampled_token_ids[:, None],
).squeeze(1)

print("Entire vocab probability at each position:\n", vocab_probabilities)
print("Chosen token IDs:", sampled_token_ids)
print("Chosen-token log probabilities:", selected_log_probs)
print("Whole-answer log probability:", selected_log_probs.sum().item())
print("Whole-answer probability:", selected_log_probs.sum().exp().item())


Entire vocab probability at each position:
 tensor([[0.0900, 0.2447, 0.6652],
        [0.6652, 0.0900, 0.2447]])
Chosen token IDs: tensor([2, 0])
Chosen-token log probabilities: tensor([-0.4076, -0.4076])
Whole-answer log probability: -0.8152120113372803
Whole-answer probability: 0.44254550337791443


## 4. 为什么抽樣／重新打分必須走一致的數值路徑？

即使參數沒有變，BF16 下「cache 生成」與「完整回答一次打分」可能因算子形狀／計算順序得到不同 logits。
如果直接把後者當作前者的 `old logp`，`current/old=1` 只能證明兩次**重新打分**相同，不能證明它就是抽樣時的策略。

這次短程使用 `scripts/grpo_rollout.py` 的明確路徑：

1. 每次 decoder 輸入保持相同的 `[回答數, 固定總長度]`，未來 token 都是 attention=0 的 padding；不使用 cache。
2. 只對當前位置的 `[回答數, 1, hidden_size]` 做 `lm_head`，取得當前 vocab logits。
3. 打分也用相同總長度 decoder，再逐位置用同形狀 `lm_head`。
4. 模型 `train()`，attention dropout=0；使用溫度 1，沒有 top-k／top-p 篩選。

因為 causal attention 遮掉未來位置，填入完整回答不應改變較早位置的數學結果；固定形狀又降低浮點算子路徑變動。
**這仍需實測**：Setonix 的 gate 直接比較所有有效 token 的 behavior logp 與 old scoring logp，再允許 backward。
它是為小型驗收選擇的可靠路徑；較快的 cache／外部 rollout engine 日後要另行驗證。


## 5. reward：如何知道完整回答對不對？

本次固定規則：回答只能有一個 `####`，其後除空白外只有一個數值；與 `target` 做精確 Decimal 比較。
正確為 1，錯誤／格式無法解析為 0。逗號、小數與負數可以解析，但沒有「隨便抓最後一個中間數字」的 fallback。

下方人工構造三段文字，只是示範檢查器；**不算模型抽樣、不算真實訓練 reward**。


In [4]:
target = row["target"]
example_completions = [
    "This is a parser demonstration, not a model response.\n#### " + target,
    "There are intermediate numbers 2, 5, 8.\n#### 999999999",
    "There are intermediate numbers 2, 5, 8, but no explicit final marker.",
]
for completion in example_completions:
    print(repr(completion))
    print(score_answer(completion, target))


'This is a parser demonstration, not a model response.\n#### 4'
{'reward': 1.0, 'predicted_answer': '4', 'target_answer': '4', 'answer_rule': 'single_final_hash_marker_exact_decimal_v1'}
'There are intermediate numbers 2, 5, 8.\n#### 999999999'
{'reward': 0.0, 'predicted_answer': '999999999', 'target_answer': '4', 'answer_rule': 'single_final_hash_marker_exact_decimal_v1'}
'There are intermediate numbers 2, 5, 8, but no explicit final marker.'
{'reward': 0.0, 'predicted_answer': None, 'target_answer': '4', 'answer_rule': 'single_final_hash_marker_exact_decimal_v1'}


## 6. reward 怎么變成更新方向？

同一題抽 $G=4$ 個回答，完整回答 reward 記為 $R_k$，$k=1,2,3,4$。
`b_all` 是這組平均值；`s` 是這組的**population 標準差**（除以 4）；`A_all,k` 是 normalized advantage。

$$
b_{\rm all}=\frac14\sum_{k=1}^{4}R_k,\qquad
s=\sqrt{\frac14\sum_{k=1}^{4}(R_k-b_{\rm all})^2},\qquad
A_{{\rm all},k}=\frac{R_k-b_{\rm all}}{s+10^{-8}}.
$$

对 `[1,1,0,0]`，均值是 0.5，標準差是 0.5，所以 advantage 約為 `[1,1,-1,-1]`。
整組同分時全部 advantage=0；不能為了產生梯度人工更改實驗 reward。


In [5]:
rewards = torch.tensor([[1.0, 1.0, 0.0, 0.0]])
b_all = rewards.mean(dim=1, keepdim=True)
s = ((rewards - b_all).square().mean(dim=1, keepdim=True)).sqrt()
A_all = (rewards - b_all) / (s + 1e-8)
assert torch.equal(A_all, group_advantages(rewards))
print("Rewards shape:", tuple(rewards.shape))
print("b_all:", b_all)
print("s:", s)
print("A_all:", A_all)
print("All-identical rewards:", group_advantages(torch.ones(1, 4)))


Rewards shape: (1, 4)
b_all: tensor([[0.5000]])
s: tensor([[0.5000]])
A_all: tensor([[ 1.,  1., -1., -1.]])
All-identical rewards: tensor([[0., 0., 0., 0.]])


## 7. old／current 比較的是同一個已抽出的 token

現在固定一批回答，`old_logp[k,t]` 是抽樣模型對第 k 個回答第 t 個 token 的 log probability，固定不求導。
`current_logp[k,t]` 是目前待更新參數對**同一 token、同一前文**的 log probability。

$$
r_{k,t}=\frac{p_{\rm current}(y_{k,t}\mid x,y_{k,<t})}
{p_{\rm old}(y_{k,t}\mid x,y_{k,<t})}
=\exp(\text{current logp}_{k,t}-\text{old logp}_{k,t}).
$$

最開始同一路徑參數相同，ratio=1。參數更新後，再對同一批 token 算 current logp，ratio 才會改變。
這是逐 token surrogate；**沒有把整段概率乘積改寫成精確相等的逐 token 和**。


In [6]:
old_logp = torch.tensor([-2.0, -2.0])
current_logp = old_logp + torch.log(torch.tensor([1.1, 0.7]))
ratio = torch.exp(current_logp - old_logp)
print("old token probabilities:", old_logp.exp())
print("current token probabilities:", current_logp.exp())
print("ratio:", ratio)


old token probabilities: tensor([0.1353, 0.1353])
current token probabilities: tensor([0.1489, 0.0947])
ratio: tensor([1.1000, 0.7000])


## 8. clipping 什麼時候停止這項激勵？

对单个 token，固定 advantage $A$，最大化的項是：

$$
\min\big(rA,\;\operatorname{clip}(r,0.8,1.2)A\big).
$$

`A>0` 时：`r>1.2` 不再繼續增大這一項；`A<0` 时：`r<0.8` 不再繼續壓低這一項。
**被裁剪後可能仍是非零數值，但对 current logp 的梯度为 0。** 梯度下降用的是負 surrogate。

下方四個獨立例子分別是：正 advantage 未裁剪／已裁剪，負 advantage 未裁剪／已裁剪。`beta=0` 用來只觀察 policy 部分。


In [7]:
old_logp = torch.full((4, 1), -2.0)
test_ratios = torch.tensor([[1.1], [1.3], [0.9], [0.7]])
current_logp = (old_logp + test_ratios.log()).requires_grad_()
test_advantages = torch.tensor([2.0, 2.0, -2.0, -2.0])
terms = grpo_loss(
    current_logp, old_logp, old_logp, test_advantages,
    torch.ones((4, 1), dtype=torch.bool), kl_coefficient=0.0,
)
terms["loss"].backward()
print("Ratios:", terms["ratios"].flatten().detach())
print("Surrogate values:", terms["per_token_surrogate"].flatten().detach())
print("Gradient of minimized loss wrt current logp:", current_logp.grad.flatten())
print("The four-answer mean contributes a factor 1/4 to each gradient.")


Ratios: tensor([1.1000, 1.3000, 0.9000, 0.7000])
Surrogate values: tensor([ 2.2000,  2.4000, -1.8000, -1.6000])
Gradient of minimized loss wrt current logp: tensor([-0.5500,  0.0000,  0.4500,  0.0000])
The four-answer mean contributes a factor 1/4 to each gradient.


## 9. 哪些 token 進 loss？

每段輸入包含 padding、prompt、completion。只有 completion 的有效 token 參與 loss；第一個 EOS 屬於模型停止生成的決策，因此也算。
EOS 後的 padding 不算；prompt 不算。沒有 EOS 的長度上限截斷回答，只保留已抽到的有效 token。

下方 `input_ids` 是 `[4,7]`：同一題的 4 個人工回答，每列前 4 個位置是 prompt（第一格是 left padding）。
`completion_starts=[4,4,4,4]` 是每列第一個回答 token 的**絕對位置**。


In [8]:
input_ids = torch.tensor([
    [0, 5, 6, 7, 8, 2, 0],
    [0, 5, 6, 7, 9, 10, 2],
    [0, 5, 6, 7, 11, 2, 0],
    [0, 5, 6, 7, 12, 13, 2],
])
attention_mask = (input_ids != 0).long()
completion_starts = torch.tensor([4, 4, 4, 4])
full_token_mask = completion_token_mask(input_ids, attention_mask, completion_starts, [2])
print("Input IDs:\n", input_ids)
print("Full token mask:\n", full_token_mask)
print("Completion lengths:", full_token_mask.sum(dim=1))
# If using full causal logits[:, :-1], predictions correspond to input_ids[:, 1:].
print("Corresponding causal-score mask:\n", full_token_mask[:, 1:])


Input IDs:
 tensor([[ 0,  5,  6,  7,  8,  2,  0],
        [ 0,  5,  6,  7,  9, 10,  2],
        [ 0,  5,  6,  7, 11,  2,  0],
        [ 0,  5,  6,  7, 12, 13,  2]])
Full token mask:
 tensor([[False, False, False, False,  True,  True, False],
        [False, False, False, False,  True,  True,  True],
        [False, False, False, False,  True,  True, False],
        [False, False, False, False,  True,  True,  True]])
Completion lengths: tensor([2, 3, 2, 3])
Corresponding causal-score mask:
 tensor([[False, False, False,  True,  True, False],
        [False, False, False,  True,  True,  True],
        [False, False, False,  True,  True, False],
        [False, False, False,  True,  True,  True]])


## 10. reference KL 是另一個固定模型的約束

`reference` 是最開始的凍結模型；`old` 是本批回答抽樣時的模型。第一批兩者參數相同，後續通常不同。
reference 不跟著每次 optimizer step 更新。

对某个已抽出的 token，记 `p=current probability`、`q=reference probability`。
$d=\log q-\log p$。我们使用 sampled-token 项：

$$
K=\exp(d)-d-1.
$$

下方直接对当前 logp 求导，结果是 $1-\exp(d)$。reference logp 固定。
这与 DeepSeekMath Eq.4 的常见实现一致，但**不是 full-vocab KL 期望的精确梯度**：固定采样 token 直接求导，没有对抽样分布求导。
K 的平均值只有在 token 按 current 分布抽样时才是该位置 KL 的无偏估计；更新后仍用旧样本、不做对应加权时，这里的平均值应读作 surrogate 诊断量。
数值实现用 `expm1(d)-d`，改善 d 接近 0 时减法消去；没有把 KL 随意 clamp 为正。


In [9]:
current_logp = torch.tensor([[-1.0]], requires_grad=True)
reference_logp = torch.tensor([[-2.0]])
d = reference_logp - current_logp
K = torch.expm1(d) - d
K.sum().backward()
print("d:", d.detach())
print("K:", K.detach())
print("Direct sampled-token gradient:", current_logp.grad)
print("At current=reference:", (torch.expm1(torch.zeros(1)) - torch.zeros(1)))


d: tensor([[-1.]])
K: tensor([[0.3679]])
Direct sampled-token gradient: tensor([[0.6321]])
At current=reference: tensor([0.])


## 11. 把每個 token 的項合成 loss

$N$ 是完整回答數；第 $k$ 個回答有效長度是 $T_k$。同一完整回答的 normalized advantage $A_k$ 给它所有有效 token。

$$
L=\frac1N\sum_{k=1}^{N}\frac1{T_k}\sum_{t=1}^{T_k}
\left[-\min(r_{k,t}A_k,\operatorname{clip}(r_{k,t},0.8,1.2)A_k)+\beta K_{k,t}\right].
$$

**先在一段回答內取 token 平均，再在回答之間取平均。** 这会让长短回答各占 1/N；不是把全批 token 一起平均。
下面一段只有 1 個有效 token，另一段 3 個；两段 loss 各占一半。


In [10]:
current_logp = torch.zeros((2, 3), requires_grad=True)
example_mask = torch.tensor([[True, False, False], [True, True, True]])
terms = grpo_loss(
    current_logp, current_logp.detach(), current_logp.detach(),
    torch.tensor([1.0, -1.0]), example_mask, kl_coefficient=0.0,
)
terms["loss"].backward()
print("Per-completion minimized loss:", terms["per_completion_loss"].detach())
print("Batch loss:", terms["loss"].item())
print("Gradient per position:\n", current_logp.grad)


Per-completion minimized loss: tensor([-1.,  1.])
Batch loss: 0.0
Gradient per position:
 tensor([[-0.5000,  0.0000,  0.0000],
        [ 0.1667,  0.1667,  0.1667]])


## 12. 真正接上一個神經網路：tiny Qwen + difference GeoRA

現在把 loss 的輸入 current logp 接到模型參數上。模型是**隨機 tiny Qwen**（1 層、hidden size=32、vocab=32），沒有預訓練，不能解 GSM8K。
本節沿用第 9 節人工 token ID，以及第 6 節人工 reward `[1,1,0,0]`，驗證反向傳播能真正改變 A/B。

FP32 在本機用來閱讀機制。Setonix 的 1.5B 模型仍用已通過的精度路徑：凍結參數 BF16，A/B/A0/B0 FP32，低秩差值 FP32，再转回 base output dtype。
`install_geora` 為小模型生成 mask/SVD 初值；載入 fresh 原模型時指定 difference，不沿用已扣殘差的 base。


In [11]:
torch.manual_seed(37)
configuration = Qwen2Config(
    vocab_size=32, hidden_size=32, intermediate_size=48,
    num_hidden_layers=1, num_attention_heads=4, num_key_value_heads=2,
    max_position_embeddings=64, attention_dropout=0.0,
    eos_token_id=2, pad_token_id=0,
)
configuration._attn_implementation = "eager"
reference_model = Qwen2ForCausalLM(configuration).float().eval()
reference_model.requires_grad_(False)
initialization_model = copy.deepcopy(reference_model)
initialization_manifest = install_geora(initialization_model, rank=2, alpha=4, rho=0.2)
initial_factors = export_adapter_state(initialization_model)
policy_model = copy.deepcopy(reference_model)
difference_manifest = dict(initialization_manifest, forward_mode="difference")
load_geora_state(policy_model, initial_factors, difference_manifest)
policy_model.train()

trainable_parameters = {
    name: parameter for name, parameter in policy_model.named_parameters()
    if parameter.requires_grad
}
print("Trainable tensors:", len(trainable_parameters))
print("Trainable names:", list(trainable_parameters))


Trainable tensors: 14
Trainable names: ['model.layers.0.self_attn.q_proj.A', 'model.layers.0.self_attn.q_proj.B', 'model.layers.0.self_attn.k_proj.A', 'model.layers.0.self_attn.k_proj.B', 'model.layers.0.self_attn.v_proj.A', 'model.layers.0.self_attn.v_proj.B', 'model.layers.0.self_attn.o_proj.A', 'model.layers.0.self_attn.o_proj.B', 'model.layers.0.mlp.gate_proj.A', 'model.layers.0.mlp.gate_proj.B', 'model.layers.0.mlp.up_proj.A', 'model.layers.0.mlp.up_proj.B', 'model.layers.0.mlp.down_proj.A', 'model.layers.0.mlp.down_proj.B']


### 12A. 固定這批人工回答，取得 old/reference scores

為與實際 rollout helper 的形狀一致，我們用 decoder 讀完整固定寬度，再對回答前一位置逐個呼叫 `lm_head`。
這個函數輸出 `[4,3]`：4 段回答，每段最多 3 個回答 token；無效位置的 logp 設為 0。

`old` 和 `reference` 只在 no_grad 下打分。`old` 分母這批之後固定；reference 參數始終凍結。


In [12]:
position_ids = attention_mask.cumsum(dim=1) - 1
position_ids = position_ids.clamp_min(0)
completion_mask = full_token_mask[:, 4:]

def score_fixed_tiny_answers(model):
    # The decoder shape is unchanged across the old, reference and current calls.
    hidden_states = model.model(
        input_ids=input_ids, attention_mask=attention_mask,
        position_ids=position_ids, use_cache=False,
    ).last_hidden_state
    selected_scores = []
    for completion_position in range(3):
        prediction_position = 3 + completion_position
        logits = model.lm_head(hidden_states[:, prediction_position:prediction_position + 1])
        full_vocab_logp = torch.log_softmax(logits[:, 0].float(), dim=-1)
        token_ids = input_ids[:, 4 + completion_position]
        selected_logp = full_vocab_logp.gather(1, token_ids[:, None]).squeeze(1)
        selected_scores.append(selected_logp)
    result = torch.stack(selected_scores, dim=1)
    return torch.where(completion_mask, result, 0.0)

with torch.no_grad():
    tiny_old_logp = score_fixed_tiny_answers(policy_model).detach()
    tiny_reference_logp = score_fixed_tiny_answers(reference_model).detach()
assert torch.equal(tiny_old_logp, tiny_reference_logp)
print("Old logp shape:", tuple(tiny_old_logp.shape))
print("Old logp:\n", tiny_old_logp)
print("Valid completion mask:\n", completion_mask)


Old logp shape: (4, 3)
Old logp:
 tensor([[-3.4370, -3.4914,  0.0000],
        [-3.6017, -3.4291, -3.5522],
        [-3.5510, -3.4353,  0.0000],
        [-3.5974, -3.4937, -3.4220]])
Valid completion mask:
 tensor([[ True,  True, False],
        [ True,  True,  True],
        [ True,  True, False],
        [ True,  True,  True]])


### 12B. loss 為 0，梯度仍然可以不是 0

初始 ratio=1 且 `[1,1,-1,-1]` 的平均是 0，所以這個人工例子的 scalar policy loss 为 0。
但是每個回答的 logp 对模型參數有**不同的導數**；正負項不一定抵消，因此 A/B 梯度可以非零。

下面真的執行一次 AdamW 更新；tiny 教學 lr=1e-4，只用来让小模型的变化容易看到。全模型 smoke 另有配置（lr=1e-6）。


In [13]:
tiny_advantages = group_advantages(torch.tensor([[1.0, 1.0, 0.0, 0.0]])).flatten()
parameters_before = {name: parameter.detach().clone() for name, parameter in trainable_parameters.items()}
tiny_current_logp = score_fixed_tiny_answers(policy_model)
tiny_terms = grpo_loss(
    tiny_current_logp, tiny_old_logp, tiny_reference_logp,
    tiny_advantages, completion_mask,
    clip_epsilon=0.2, kl_coefficient=0.001,
)
optimizer = torch.optim.AdamW(
    trainable_parameters.values(), lr=1e-4,
    betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0,
)
optimizer.zero_grad(set_to_none=True)
tiny_terms["loss"].backward()
gradient_norm = torch.nn.utils.clip_grad_norm_(list(trainable_parameters.values()), max_norm=1.0)
optimizer.step()
changed_names = [
    name for name, parameter in trainable_parameters.items()
    if not torch.equal(parameter, parameters_before[name])
]
print("Initial loss:", tiny_terms["loss"].item())
print("Initial ratio on valid tokens:", tiny_terms["ratios"][completion_mask].detach())
print("Initial reference KL:", tiny_terms["kl"].item())
print("Gradient norm before clipping:", gradient_norm.item())
print("Changed A/B tensor count:", len(changed_names))
assert gradient_norm.item() > 0 and len(changed_names) > 0


Initial loss: 0.0
Initial ratio on valid tokens: tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])
Initial reference KL: 0.0
Gradient norm before clipping: 0.1827220916748047
Changed A/B tensor count: 14


### 12C. 更新之后，old 分母与 reference 仍固定

现在 current 模型参数改变，重新给同一批固定人工回答打分。我们只观察 ratio／loss／KL 的变化。
这是机械演示，不证明模型解题更好；真实训练的 reward 来自**更新后新抽出的回答**，也不要求每一步单调上升。


In [14]:
with torch.no_grad():
    tiny_updated_logp = score_fixed_tiny_answers(policy_model)
    tiny_after = grpo_loss(
        tiny_updated_logp, tiny_old_logp, tiny_reference_logp,
        tiny_advantages, completion_mask,
    )
print("Current/old ratios on the same valid tokens:", tiny_after["ratios"][completion_mask])
print("Loss on the same artificial answers:", tiny_after["loss"].item())
print("Reference KL on the same artificial answers:", tiny_after["kl"].item())


Current/old ratios on the same valid tokens: tensor([1.0009, 1.0000, 1.0004, 1.0004, 1.0000, 1.0002, 0.9998, 0.9996, 0.9999,
        0.9998])
Loss on the same artificial answers: -0.00023436546325683594
Reference KL on the same artificial answers: 7.533435564255342e-08


## 13. 查看 Setonix 的真實 GSM8K 結果

这一节**只读取已保存的报告**，不在 notebook 启动 GPU 任务，也不把上面的人工 reward 当成真实实验。

真實驗收應保存：問題／抽樣回答、答案提取、reward、advantage、EOS／長度上限、behavior／old logp 差異、ratio、KL、有限非零梯度、A/B 更新及用量。
如果尚未有真实报告，下面明确显示未完成；完整训练、LoRA／GeoRA 分数对照是后续阶段。

本次 strict reward=0 也包括格式不符，例如算出 4 但缺標記，或寫成 `#### 4 Scoops`；第二題有 3 個截斷回答。不能把 reward 全部解讀為數學正誤；正式對照固定相同規則並另報格式失敗／截斷。


In [15]:
report_directory = PROJECT_DIRECTORY / "reports" / "grpo"
real_report_paths = sorted(
    (path for path in report_directory.glob("*.json") if path.stem.isdigit()),
    key=lambda path: int(path.stem),
) if report_directory.exists() else []
real_reports = []
for path in real_report_paths:
    candidate = json.loads(path.read_text())
    if candidate.get("stage") == "gsm8k_first_real_grpo_update":
        real_reports.append((path, candidate))

if not real_reports:
    print("尚未有可读取的 Setonix 真實 GRPO 報告。上面只有 CPU 教學和機械檢查。")
else:
    path, candidate = real_reports[-1]
    print("Report:", path)
    print("Status:", candidate.get("status"))
    print("Stage:", candidate.get("stage"))
    print("Optimizer steps:", candidate.get("optimizer_steps", 0))
    print("Checks passed:", sum(check.get("passed", False) for check in candidate.get("checks", [])))
    selected_rollout = candidate.get("selected_rollout")
    if not selected_rollout:
        print("报告还没有选中可更新的一批真实回答；请查看 attempted_batches 和 failed check。")
    else:
        print("Raw rewards:", selected_rollout.get("rewards"))
        print("Advantages:", selected_rollout.get("advantages"))
        for answer in selected_rollout.get("answers", []):
            print("\nQuestion ID:", answer["question_id"])
            print("Question:", answer["question"])
            print("Sampled completion:\n" + answer["completion"])
            print("Target / extracted:", answer["target_answer"], "/", answer["predicted_answer"])
            print("Reward / training reward:", answer["reward"], "/", answer.get("training_reward"))
            print("Truncated / token count:", answer["truncated"], "/", answer["token_count"])


Report: /Users/tianbaochen/geora-repro/reports/grpo/50601905.json
Status: passed
Stage: gsm8k_first_real_grpo_update
Optimizer steps: 1
Checks passed: 20
Raw rewards: [[0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0]]
Advantages: [-0.5773502588272095, -0.5773502588272095, -0.5773502588272095, 1.732050895690918, 0.0, 0.0, 0.0, 0.0]

Question ID: train:01480
Question: Oli's banana split has 4 scoops of ice cream while Victoria has twice more scoops of ice cream than Oli's. How many more scoops of ice cream does Victoria have than Oli?
Sampled completion:
First, let's determine how many scoops of ice cream are in Victoria's banana split.

1. Oli's banana split has 4 scoops of ice cream.
2. Victoria's banana split has twice as many scoops as Oli's. Let's calculate that:
   \[
   \text{Number of scoops in Victoria's banana split} = 2 \times 4 = 8
   \]
3. Now we need to find out how many more scoops of ice cream Victoria has than Oli:
   \[
   \text{Difference in scoops} = 8 - 4 = 4
   \]

Theref

## 14. 到這裡，應該有的心智模型

1. 檢查器給**完整回答**一個 reward；不需要它對參數可微分。
2. 組內比較把 reward 變成每段回答的固定 advantage。
3. current 模型對已抽出的每個 token 算 logp；ratio、clipping 和 reference KL 構成可微分 loss。
4. backward 沿 logits → Transformer → GeoRA 的 A/B 求導；optimizer 只改 A/B。
5. 下一批需要重新抽樣回答；分數是否提高要另外用固定評估資料量測。

**我們剛驗證／演示的是訓練管線能否正確產生更新。沒有完成論文分數復現。**

可選練習：把第 6 節 reward 改成 `[1,0,0,0]`，手算再运行；第 8 節把 ratio 改為 `[0.7,1.3]` 看 clipping 在不利方向是否仍保留恢復梯度。
